# Hierarchical Clustering

In this programming lesson, you are going to learn to implement hierarchical clustering algorithms you have learnt using `sklearn`  library.

With the increasing amount of biological data we are gathering every day, machine learning algorithm are indispensable for analyzing those large amount of data. Clustering algorithm are generally used to find some pattern in the dataset and is used in the exploratory data analysis phase where people try to find the underlying pattern in the dataset and make inference. Clustering has proven to be helpful in biological data.

In this lesson we are going to use a gene dataset which contains the gene information of number of persons. In that dataset, we will cluster them according to the proximity of gene. If we can find certain gene that are responsible for some disease, then we can prevent those disease in a patient even before it starts to show symptoms.

Because we are trying to find the patterns in a large dataset, some applications like __BIRCH__ will work efficiently and others does not. In this programming we will also compare the performances of those algorithm too.

__POINTS: (TO REMOVE)__

1. Data clustering is good for exploratory data analysis

2. We show BIRCH is better than most for large dataset

3. Compare running time, memory usage, clusterign quality, stability and scalability

## Download Dataset

In [ ]:
!wget -O output.tar.gz https://archive.ics.uci.edu/ml/machine-learning-databases/00401/TCGA-PANCAN-HiSeq-801x20531.tar.gz
!tar -xzvf output.tar.gz>/dev/null

--2022-04-22 16:11:05--  https://archive.ics.uci.edu/ml/machine-learning-databases/00401/TCGA-PANCAN-HiSeq-801x20531.tar.gz
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 72856320 (69M) [application/x-httpd-php]
Saving to: ‘output.tar.gz’

output.tar.gz       100%[===================>]  69.48M  19.7MB/s    in 4.5s    

2022-04-22 16:11:10 (15.4 MB/s) - ‘output.tar.gz’ saved [72856320/72856320]



## Loading the dataset

Let's read the `csv`  using the pandas library.

In [ ]:
import time

import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('./TCGA-PANCAN-HiSeq-801x20531/data.csv', index_col=0)

Let's visualize the head of the dataset.

In [ ]:
print(f'Shape of the dataset: {data.shape}')
data.head()

Shape of the dataset: (801, 20531)


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,0.0,...,4.926711,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0
sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,0.0,...,4.593372,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0
sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,0.0,...,5.125213,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0
sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,0.0,...,6.076566,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0
sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,0.0,...,5.996032,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0


The dataset contains $20531$ columns and $801$ rows. Each row represents the gene information of a person with $20531$ number of features or attributes.

### Data details

Because in this phase we are doing the EDA, we try to find various patterns in the data. Pandas dataframe contains methods to find various information of data. Let's use the `describe` method to find the statistical properties like mean, media, etc.

In [ ]:
data.describe()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
count,801.000000,801.000000,801.000000,801.000000,801.000000,801.0,801.000000,801.000000,801.000000,801.000000,...,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000
mean,0.026642,3.010909,3.095350,6.722305,9.813612,0.0,7.405509,0.499882,0.016744,0.013428,...,5.896573,8.765891,10.056252,4.847727,9.741987,11.742228,10.155271,9.590726,5.528177,0.095411
std,0.136850,1.200828,1.065601,0.638819,0.506537,0.0,1.108237,0.508799,0.133635,0.204722,...,0.746399,0.603176,0.379278,2.382728,0.533898,0.670371,0.580569,0.563849,2.073859,0.364529
min,0.000000,0.000000,0.000000,5.009284,8.435999,0.0,3.930747,0.000000,0.000000,0.000000,...,2.853517,6.678368,8.669456,0.000000,7.974942,9.045255,7.530141,7.864533,0.593975,0.000000
25%,0.000000,2.299039,2.390365,6.303346,9.464466,0.0,6.676042,0.000000,0.000000,0.000000,...,5.454926,8.383834,9.826027,3.130750,9.400747,11.315857,9.836525,9.244219,4.092385,0.000000
50%,0.000000,3.143687,3.127006,6.655893,9.791599,0.0,7.450114,0.443076,0.000000,0.000000,...,5.972582,8.784144,10.066385,5.444935,9.784524,11.749802,10.191207,9.566511,5.218618,0.000000
75%,0.000000,3.883484,3.802534,7.038447,10.142324,0.0,8.121984,0.789354,0.000000,0.000000,...,6.411292,9.147136,10.299025,6.637412,10.082269,12.177852,10.578561,9.917888,6.876382,0.000000
max,1.482332,6.237034,6.063484,10.129528,11.355621,0.0,10.718190,2.779008,1.785592,4.067604,...,7.771054,11.105431,11.318243,9.207495,11.811632,13.715361,11.675653,12.813320,11.205836,5.254133


## Standardization

As we have seen that for each feature the range of values are different which can have undesired side effects on the proximity measure. To solve this, we are going to apply a simple standardization technique in the training dataset. We will use a simple [standardscaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) class for this operation.

In [ ]:
from sklearn.preprocessing import None

scaler = None

# fit the scaler and transform the data
X = None

Let's see the first five rows in the scaled dataset. Because the scaled variable `X` is numpy array, let's convert it to dataframe just for this visualization.

In [ ]:
X_pd = pd.DataFrame(X, columns=data.columns)
X_pd.head()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
0,-0.194799,-0.828030,0.159800,-1.948277,1.221575,0.0,-0.207968,0.180910,-0.125376,-0.065633,...,-1.300200,-0.921755,-0.877838,0.996247,-1.166072,0.389441,-0.869566,-1.187938,-0.116483,-0.261901
1,-0.194799,-2.015017,-1.415042,1.353109,-0.376518,0.0,-0.532222,-0.983088,-0.125376,-0.065633,...,-1.747076,-2.392213,-0.831893,0.591649,-2.549598,1.391628,0.623551,-0.342277,-1.656889,-0.261901
2,-0.194799,0.417348,1.156735,0.249807,0.112832,0.0,-0.391297,-0.092995,-0.125376,-0.065633,...,-1.034088,-1.059669,2.248803,0.232601,0.317880,-4.025621,-0.632381,0.886860,-1.855264,-0.261901
3,-0.194799,0.543888,1.326182,-0.099053,0.755741,0.0,0.395348,-0.127831,-0.125376,-0.065633,...,0.241299,0.044905,0.224956,1.719725,-0.263847,-0.521747,-0.793609,0.166174,-1.078941,-0.261901
4,-0.194799,-0.295955,-0.257107,-0.286413,-0.148843,0.0,-0.757118,-0.273165,-0.125376,-0.065633,...,0.133335,0.208252,0.837739,0.979924,0.196645,0.268992,-1.615841,-0.229878,-0.201588,-0.261901


In [ ]:
X_pd.describe()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
count,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,801.0,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,...,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02,8.010000e+02
mean,7.352975e-17,-4.920464e-17,5.250343e-16,1.298725e-16,-2.066595e-16,0.0,-3.579464e-16,4.463069e-17,4.010525e-16,-2.972030e-16,...,-1.142102e-15,-7.258724e-16,-2.391553e-15,4.546232e-17,1.677532e-15,-1.810384e-15,8.769514e-16,2.731204e-16,-2.564879e-16,2.171934e-16
std,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,0.0,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,...,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00,1.000625e+00
min,-1.947993e-01,-2.508929e+00,-2.906607e+00,-2.683220e+00,-2.721366e+00,0.0,-3.137355e+00,-9.830880e-01,-1.253755e-01,-6.563299e-02,...,-4.079532e+00,-3.463046e+00,-3.658700e+00,-2.035799e+00,-3.311774e+00,-4.025621e+00,-4.524475e+00,-3.063356e+00,-2.380723e+00,-2.619014e-01
25%,-1.947993e-01,-5.931866e-01,-6.619973e-01,-6.562442e-01,-6.897104e-01,0.0,-6.586345e-01,-9.830880e-01,-1.253755e-01,-6.563299e-02,...,-5.920735e-01,-6.338043e-01,-6.073885e-01,-7.210434e-01,-6.395484e-01,-6.364197e-01,-5.493656e-01,-6.149231e-01,-6.927610e-01,-2.619014e-01
50%,-1.947993e-01,1.106410e-01,2.972590e-02,-1.040266e-01,-4.348466e-02,0.0,4.027376e-02,-1.117158e-01,-1.253755e-01,-6.563299e-02,...,1.018984e-01,3.028065e-02,2.673326e-02,2.507971e-01,7.972069e-02,1.130606e-02,6.193737e-02,-4.297362e-02,-1.493600e-01,-2.619014e-01
75%,-1.947993e-01,7.270983e-01,6.640633e-01,4.951942e-01,6.493447e-01,0.0,6.469035e-01,5.692881e-01,-1.253755e-01,-6.563299e-02,...,6.900345e-01,6.324578e-01,6.404919e-01,7.515766e-01,6.377510e-01,6.502323e-01,7.295516e-01,5.805917e-01,6.505011e-01,-2.619014e-01
max,1.064381e+01,2.688263e+00,2.787148e+00,5.336962e+00,3.046117e+00,0.0,2.991012e+00,4.482225e+00,1.324467e+01,1.981568e+01,...,2.512935e+00,3.881123e+00,3.329434e+00,1.830881e+00,3.878901e+00,2.945186e+00,2.620415e+00,5.718915e+00,2.739437e+00,1.416059e+01


Here, you can see that all the features now has zero mean and standard deviation of one. 

## Fit the model

In this section, we will fit agglomerative clustering models with different linkage method as well as the BIRCH algorithm. We will compare the performance of these algorithm with different number of clusters (cut at the dendogram).

### Single Linkage

Let's fit a hierarchical clustering method with single linkage.

In [ ]:
# import the model class
from sklearn.cluster import AgglomerativeClustering

Let's create the single linkage agglomerative clustering algorithm. We will split the dataset into 10 number of clusters.

In [ ]:
# create the model
K = 10
single_linkage = None

Now, we will fit the model with the dataset `X`.

In [ ]:
time_st = time.time()
single_linkage.fit(X)

# time taken to fit single linkage algorithm
time_taken_sl = time.time() - time_st

print(f'The single linkage algorithm took {time_taken_sl:.2f} seconds to fit.')

The single linkage algorithm took 10.75 seconds to fit.


### Complete Linkage

Let's fit the agglomerative algorithm with complete linkage.

In [ ]:
# create the model with complete linkage
K = 10
complete_linkage = None

Now, lets fit the training data with this model.

In [ ]:
time_st = time.time()
complete_linkage.fit(X)

# time taken to fit single linkage algorithm
time_taken_cl = time.time() - time_st

print(f'The complete linkage algorithm took {time_taken_cl:.2f} seconds to fit.')

The complete linkage algorithm took 6.98 seconds to fit.


### Average Linkage

Now, we will fit the agglomerative clustering algorithm with average linkage.

In [ ]:
# create the model with Average linkage
K = 10
average_linkage = None

Let's fit the model in the dataset `X`.

In [ ]:
time_st = time.time()

average_linkage.fit(X)

# time taken to fit single linkage algorithm
time_taken_al = time.time() - time_st

print(f'The average linkage algorithm took {time_taken_al:.2f} seconds to fit.')

The average linkage algorithm took 6.73 seconds to fit.


### BIRCH model

Let's fit the model for a specific number of clusters and see the performance difference between BIRCH algorithm.

In [ ]:
# import the birch model from sklearn
from sklearn.cluster import None

# threshold
T = 0.6

# branching factor
B = 100

birch = None

Now, fitting the model `birth` on dataset `X`,

In [ ]:
time_st = time.time()

birch.fit(X)

# time taken to fit single linkage algorithm
time_taken_birch = time.time() - time_st

print(f'The BIRCH algorithm took {time_taken_birch:.2f} seconds to fit.')

The BIRCH algorithm took 10.38 seconds to fit.
